# 09.02 · Subagents (supervisor-style orchestration)

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 09.01; Module 02 (tools, `ToolRuntime`); Module 08.03 (interrupts). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Wrap** specialist agents as tools for a coordinator, first by hand (the docs pattern), then with `mastery.agents.subagent_tool`.
2. **Control** what a subagent sees (isolated query vs forked history) and what it returns, and measure the token cost of each.
3. **Dispatch** through one `task(agent_name, description)` tool with an enum of agent names.
4. **Propagate** an `interrupt()` from inside a subagent's tool to the outer agent, and resume it.
5. **Migrate** a `langgraph-supervisor` setup to subagents.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents) | Tool per agent vs single dispatch tool, sync vs async, context engineering (inputs/outputs) |
| [Build a personal assistant with subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant) | A full supervisor built with `create_agent` + wrapped specialists, HITL on the subagents' tools |
| [Migrate from langgraph-supervisor](https://docs.langchain.com/oss/python/migrate/langgraph-supervisor) | `create_supervisor` → `create_agent` + `@tool` wrappers; interrupt propagation rules; `output_mode` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the shared desk (Lesson 09.01 Part 2).
from langgraph.checkpoint.memory import InMemorySaver
from mastery.agents import UsageMeter
from mastery.agents.support import (COORDINATOR_PROMPT, SupportDesk, evaluate, faq_search, load_conversations,
                                    make_specialists, make_support_tools, offline_model, run_conversation)

desk = SupportDesk.load()
tools = make_support_tools(desk, faq_search())
specialists = make_specialists(tools)          # name, description, system prompt, tools per domain
by_id = {c["id"]: c for c in load_conversations()}
print({name: [t.name for t in spec.tools] for name, spec in specialists.items()})

**You should now see** the three specialists and their tools: billing (`lookup_invoices`, `issue_refund`), technical (`search_faq`, `check_service_status`), account (3 tools).

---
## Part 1 · The docs pattern, by hand

A subagent is just an agent **called inside a tool**. The coordinator sees one tool per specialist; the tool's description is how it decides whom to call. Subagents get **no checkpointer** (they inherit the coordinator's at runtime, Part 4).

In [ ]:
# 🧪 Offline: two specialists wrapped with @tool, exactly like the Subagents docs.
from langchain.agents import create_agent
from langchain.tools import tool

billing_agent = create_agent(offline_model(), tools=list(specialists["billing"].tools),
                             system_prompt=specialists["billing"].system_prompt, name="billing")
technical_agent = create_agent(offline_model(), tools=list(specialists["technical"].tools),
                               system_prompt=specialists["technical"].system_prompt, name="technical")


@tool("billing_agent", description="The billing specialist: invoices, charges, refunds.")
def call_billing(query: str) -> str:
    result = billing_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


@tool("technical_agent", description="The technical specialist: API errors, sync, webhooks, status.")
def call_technical(query: str) -> str:
    result = technical_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


coordinator = create_agent(offline_model(), tools=[call_billing, call_technical], system_prompt=COORDINATOR_PROMPT,
                           name="coordinator", checkpointer=InMemorySaver())
meter = UsageMeter()
out = coordinator.invoke({"messages": [{"role": "user", "content": by_id["S14"]["turns"][0]}]},
                         {"configurable": {"thread_id": "by-hand"}, "callbacks": [meter]})
for m in out["messages"]:
    if m.type == "ai" and m.tool_calls:
        for c in m.tool_calls:
            print(f"coordinator → {c['name']}({c['args']['query'][:60]}...)")
    elif m.type == "tool":
        print(f"  {m.name} ← {m.content[:80]}...")
print("final:", out["messages"][-1].content[:90], "...")
print(meter.agents())

**You should now see** the coordinator calling `billing_agent` and `technical_agent` **in the same step** (two tool calls in one AI message), each with only **its** part of the request (the refund clause; the 429 clause plus `(customer C-1001)`), both results coming back (`Refunded $49.00 ...` and the `API errors and rate limits` FAQ passage), a final answer combining them, and 2 model calls each for `coordinator`, `billing` and `technical`.

### What just happened?

`create_agent` runs the tool calls of one AI message **in parallel**, so the two specialists worked at the same time; each made 2 calls (pick a tool, answer). The coordinator made 2 calls (delegate, then answer). That final coordinator call is the "extra" call subagents always pay: the specialists' answers go **back** to the coordinator, who writes the reply. That's the price of control.

---
## Part 2 · What does the subagent see? Isolated query vs forked history

`subagent_tool(agent, name=..., description=..., context=...)` is Part 1's wrapper plus two choices from the docs' *context engineering* section:

- `context="query"` (default, **isolated**): the subagent sees only what the coordinator wrote.
- `context="history"` (**forked**): it also gets the coordinator's conversation so far (via `ToolRuntime`, as a transcript).

The classic bug: the customer gave their ID two turns ago, the coordinator writes a short task without it, and the specialist asks again.

In [ ]:
# 🧪 Offline: a coordinator that "forgets" the customer ID when it writes the task.
from langchain_core.messages import AIMessage, HumanMessage
from mastery.agents import subagent_tool
from mastery.testing import ScriptedChatModel

history = [HumanMessage("I'm C-1001, can you list my invoices?"),
           AIMessage("You have INV-1001, INV-1002 and INV-1003."),
           HumanMessage("Please refund INV-1003 ($49), it was a duplicate.")]
forgetful_call = AIMessage(content="", tool_calls=[{"name": "billing_agent", "id": "t1", "type": "tool_call",
                                                    "args": {"query": "Refund INV-1003 ($49), duplicate charge."}}])
for context in ("query", "history"):
    desk.reset()
    sub = specialists["billing"].build(offline_model())
    wrapped = subagent_tool(sub, name="billing_agent", description="Billing specialist.", context=context)
    boss = create_agent(ScriptedChatModel(responses=[forgetful_call, "(coordinator answers)"]), tools=[wrapped],
                        name="coordinator")
    meter = UsageMeter()
    out = boss.invoke({"messages": history}, {"callbacks": [meter]})
    print(f"{context:<8} → {out['messages'][-2].content[:70]:<72} billing input tokens: {meter.agents()['billing']['input_tokens']}")

**You should now see**

- `query` → `Could you give me your customer ID? It looks like C-1001.` (the specialist never saw the ID), with fewer billing input tokens;
- `history` → `Refunded $49.00 for INV-1003 ...`, with more input tokens (the transcript rides along).

### What just happened?

Isolation is cheaper and keeps specialists focused, but the **coordinator** then owns context: its prompt must say "include the customer ID" (ours does, `COORDINATOR_PROMPT`). Forking fixes the bug for every specialist at once, at a token cost that grows with the conversation. A middle way the docs show: pass only selected state (here: the customer ID) through `ToolRuntime.state`.

**Outputs** have the same choice: `output="last"` (default) returns the specialist's final message; `output="all"` returns a transcript of its run (like `langgraph-supervisor`'s `output_mode="full_history"`). The docs also show returning `Command(update={...})` from the wrapper to write extra keys into the coordinator's state.

---
## Part 3 · One dispatch tool: `task(agent_name, description)`

With many specialists, one tool per agent bloats the coordinator's tool list. The docs' alternative is a **single** `task` tool whose `agent_name` is constrained to the registered names (an enum), with the list in its description.

In [ ]:
# 🧪 Offline: the task tool's schema, then the same S14 request through it.
import json

from mastery.agents import task_tool
from mastery.agents.support import DESCRIPTIONS, build_subagents_desk

built = {name: spec.build(offline_model()) for name, spec in specialists.items()}
task = task_tool({name: (agent, DESCRIPTIONS[name]) for name, agent in built.items()})
schema = task.args_schema.model_json_schema()
print(task.description)
print("agent_name enum:", schema["$defs"]["AgentName"]["enum"])

desk.reset()
dispatch_desk = build_subagents_desk(offline_model, tools, checkpointer=InMemorySaver(), dispatch="task")
out = dispatch_desk.invoke({"messages": [{"role": "user", "content": by_id["S14"]["turns"][0]}]},
                           {"configurable": {"thread_id": "task-1"}})
print([(c["name"], c["args"]["agent_name"]) for m in out["messages"] if m.type == "ai" for c in m.tool_calls])

**You should now see** the description listing `billing`, `technical`, `account` with one line each, `agent_name enum: ['billing', 'technical', 'account']`, and two `task` calls: `('task', 'billing')`, `('task', 'technical')`.

> Same behaviour, one tool. Use per-agent tools when each specialist needs its **own** argument schema or description details; use `task` when you have many similar specialists or add them at runtime (the docs' "discovery tool" variant lists them on demand).

---
## Part 4 · Interrupts inside subagents

`issue_refund` calls `interrupt()` above $50. Here the call happens **inside** the billing subagent, **inside** a tool of the coordinator. The migration guide's two rules make it work:

1. Only the **outermost** agent gets a checkpointer (subagents inherit it at runtime).
2. Every outer call passes a `thread_id`.

In [ ]:
# 🧪 Offline: a $199 refund (S02) pauses the coordinator; we count calls before and after resuming.
from langgraph.types import Command

desk.reset()
app = build_subagents_desk(offline_model, tools, checkpointer=InMemorySaver())
meter = UsageMeter()
config = {"configurable": {"thread_id": "refund-199"}, "callbacks": [meter]}
out = app.invoke({"messages": [{"role": "user", "content": by_id["S02"]["turns"][0]}]}, config)
pause = out["__interrupt__"][0]
print("paused:", {k: pause.value[k] for k in ("type", "invoice_id", "amount", "threshold")})
print("calls before resume:", meter.agents())

out = app.invoke(Command(resume="approve"), config)
print("final:", out["messages"][-1].content)
print("calls after resume :", meter.agents())
print("refunds:", desk.refunds, "| tool log:", desk.tools_called())

**You should now see** `paused: {'type': 'refund_approval', 'invoice_id': 'INV-2001', 'amount': 199.0, 'threshold': 50.0}`; before resuming `coordinator: 1, billing: 1` calls; then `Refunded $199.00 for INV-2001 (approved by billing manager) ...` and `coordinator: 2, billing: 2`; one refund record and `['issue_refund']` logged **once**.

### What just happened?

The interrupt bubbled up through the tool and both agents and paused the **coordinator's** thread. On resume, the billing subagent did **not** call its model again to re-decide (still 1 call before its final answer): it resumed from the checkpoint it wrote under the parent's thread. Only the tool function itself re-ran from the top, which is why `issue_refund` logs **after** the interrupt (Lesson 08.03: code before `interrupt()` runs twice).

> `get_state(config, subgraphs=True)` does **not** show subagents called inside tools (the docs say so): they aren't graph nodes. Inspect them through the meter, traces (LangSmith), or what the tool returns.

---
## Part 5 · The whole test set, per agent

In [ ]:
# 🧪 Offline: 15 conversations, isolated vs forked context, with per-agent counts.
rows = []
for context in ("query", "history"):
    meter = UsageMeter()
    report = evaluate(build_subagents_desk(offline_model, tools, checkpointer=InMemorySaver(), context=context),
                      list(by_id.values()), desk, meter=meter)
    per_agent = {name: counts["model_calls"] for name, counts in meter.agents().items()}
    rows.append({"context": context, "passed": f"{report['passed']}/15", "calls": meter.model_calls,
                 "tokens": f"{meter.total_tokens:,}", "calls per agent": per_agent})
table(rows, ["context", "passed", "calls", "tokens", "calls per agent"])

**You should now see** `15/15` both times and **71** calls (`coordinator: 35`, `billing: 20`, `technical: 10`, `account: 6`), with `history` using more tokens than `query` (about 27.9K vs 26.6K here; the gap grows with longer conversations, because the transcript is sent to every specialist call).

Half of all calls are the coordinator's: 2 per turn, on every turn. That is the number to beat in Lesson 09.03.

---
## Part 6 · Migrating from `langgraph-supervisor`

`langgraph-supervisor` is **no longer actively maintained** (verified 2026-10-12). The mapping from the migration guide:

| `langgraph-supervisor` | Subagents (`create_agent`) |
|---|---|
| `create_supervisor([agents], model=..., prompt=...)` | `create_agent(model, tools=[wrapped specialists], system_prompt=...)` |
| Worker agents as graph nodes | Worker agents called inside `@tool` functions |
| `create_handoff_tool` | A custom `@tool` that calls `subagent.invoke(...)` |
| `output_mode="last_message"` / `"full_history"` | Format the result in the wrapper (`output="last"` / `"all"`) |
| Nested supervisors | A middle-tier agent wrapped as a tool (or flatten to one supervisor) |
| `.compile(checkpointer=...)` | `checkpointer=` on the **outermost** agent only |

    ---
    ## Part 7 · 💸 Subagents with real models

> 💸 Calls `gpt-oss-20b` several times (multi-agent = many small calls), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: coordinator + specialists on gpt-oss-20b, a two-domain request (S14) and a $199 refund (S02).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    def live_model(role):
        return get_model("fast", temperature=0, reasoning_effort="low")

    live = build_subagents_desk(live_model, tools, checkpointer=InMemorySaver())
    meter = UsageMeter()
    report = evaluate(live, [by_id["S14"], by_id["S02"]], desk, meter=meter)
    for row in report["rows"]:
        o = row["outcome"]
        print(row["id"], "✅" if row["passed"] else "❌", o["tools"], "pauses:", o["interrupts"], "|", o["answer"][:140])
    print(meter.agents())
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) S14 with `issue_refund` and `search_faq` and an answer mentioning the refund and `Retry-After`, and S02 with `pauses: 1` (approved by the scripted manager) and a $199 refund; per-agent counts similar to the offline ones (a real coordinator sometimes calls one specialist twice). A ❌ usually means the coordinator dropped the customer ID or the amount from the task: compare with Part 2. Without a key: `⏭️ Skipped`.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| The interrupt never reaches you; the tool raises or the run hangs | A subagent compiled **with its own** checkpointer, or no `thread_id` on the outer call | Checkpointer on the outermost agent only; always pass `thread_id` |
| Specialist asks for information the user already gave | Isolated context and a coordinator that writes short tasks | Say "include the customer ID" in the coordinator prompt, or `context="history"`, or pass selected state |
| A side effect happens twice after approval | Code **before** `interrupt()` in the tool re-runs on resume | Do side effects after the decision (like `issue_refund`), or make them idempotent |
| `get_state(subgraphs=True)` shows no subagent | Subagents inside tools aren't subgraph nodes | Use `UsageMeter`, LangSmith traces, or return what you need from the wrapper |
| Coordinator invents an agent name with `task` | Free-text `agent_name` | Use an enum schema (`task_tool` does) |
| Live run hits `429` | Many small calls per turn on the free tier | Fewer conversations, or wait; `get_model` retries |

## ⚠️ Production notes

> ⚠️ **Production note: the coordinator is your hottest path.**
> It runs twice per turn, on every turn. Keep its prompt short, its tool descriptions precise, and consider a stronger model **only** there.

> ⚠️ **Production note: specialists are units you can test alone.**
> Because each specialist is a plain agent with its own tools, you can run its own eval set without the coordinator (`specialists["billing"].build(model).invoke(...)`). Do that before blaming the orchestration.

## 🏋️ Exercises

1. **(Easy)** Add the account specialist to Part 1's hand-written coordinator and run S11 (email change).
2. **(Medium)** Write a wrapper that passes **only the customer ID** from the coordinator's state instead of the whole history (hint: `ToolRuntime`, a regex over `runtime.state["messages"]`). Re-run Part 2's experiment.
3. **(Medium)** Use `output="all"` for the technical specialist and look at what the coordinator receives. When would you want that?
4. **(Stretch)** Nest: make a "billing team" agent whose tools are a refunds subagent and an invoices subagent, wrap it as a tool for the top coordinator, and check that a $199 refund still interrupts at the top.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
account_agent = specialists["account"].build(offline_model())

@tool("account_agent", description="The account specialist: passwords, locks, email changes.")
def call_account(query: str) -> str:
    return account_agent.invoke({"messages": [{"role": "user", "content": query}]})["messages"][-1].content

coordinator = create_agent(offline_model(), tools=[call_billing, call_technical, call_account],
                           system_prompt=COORDINATOR_PROMPT, name="coordinator", checkpointer=InMemorySaver())
print(run_conversation(coordinator, by_id["S11"], desk)["answer"])
```

**2.**
```python
import re
from langchain.tools import ToolRuntime

@tool("billing_agent", description="Billing specialist.")
def billing_with_id(query: str, runtime: ToolRuntime) -> str:
    text = " ".join(m.text for m in runtime.state["messages"] if m.type == "human")
    ids = re.findall(r"C-\d{4}", text)
    task = f"{query} (customer {ids[-1]})" if ids and ids[-1] not in query else query
    return sub.invoke({"messages": [{"role": "user", "content": task}]})["messages"][-1].content
```
Fewer tokens than `history`, same fix for this bug.

**3.** `subagent_tool(built["technical"], name="technical_agent", description="...", output="all")`: the coordinator gets `tool search_faq: ...` lines too. Useful when the coordinator must cite sources or check the specialist's work; it costs coordinator tokens on every later call.

**4.** Build `refunds = Specialist(..., tools=(tools["billing"][1],)).build(offline_model())`, wrap it, build `billing_team = create_agent(offline_model(), tools=[...])` **without** a checkpointer, wrap that as a tool for the top coordinator **with** the checkpointer. The interrupt surfaces at the top (the migration guide's nested example). Note the offline model only knows the desk's prompts, so give the middle tier a specialist-style prompt or script it with `ScriptedChatModel`.
</details>

## ✅ Checkpoint

- [ ] I wrapped specialists as tools by hand and with `subagent_tool`.
- [ ] I can explain isolated vs forked context and measured its token cost.
- [ ] I used a single `task` tool with an enum of agent names.
- [ ] A refund over $50 inside a subagent paused the coordinator and resumed without repeating the decision.
- [ ] I can map `create_supervisor` code to the subagents pattern.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read the [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents) page's **async** section and note in [`docs/research-log.md`](../../docs/research-log.md): which three tools the docs use for background subagent jobs, and when you'd pick async over this lesson's sync subagents.

## Summary

- Subagent = an agent called inside a tool; the coordinator decides every step and writes every reply (2 calls per turn).
- Context is a choice: isolated (`query`) is cheap but the coordinator must pass what matters; forked (`history`) is robust but grows with the conversation.
- `task(agent_name, description)` with an enum scales to many specialists.
- One checkpointer at the top + a `thread_id` → interrupts deep inside subagents pause and resume the whole system.
- `langgraph-supervisor` → `create_agent` + wrapped specialists.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 09.03 · Handoffs** → [`09-03-handoffs.ipynb`](09-03-handoffs.ipynb).

[Module 09 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)